# Playground Series S6E4 — Predicting Irrigation Need

**Task**: Multi-class classification (`Low` / `Medium` / `High`)  
**Metric**: Balanced Accuracy  
**Challenge**: Severe class imbalance — `High` is only ~3.3% of training data

**Pipeline**:
1. Load data (auto-detects Kaggle vs local path)
2. EDA — missing values, class balance, distributions, correlations
3. Feature engineering — agronomic domain features
4. 5-fold stratified CV with LightGBM, XGBoost, CatBoost
5. Optuna hyperparameter tuning
6. Ensemble weight search on OOF predictions
7. Submission

In [ ]:
# Install missing packages (runs silently on Kaggle)
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

try:
    import optuna
except ImportError:
    _install('optuna')

print('Environment ready')

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── GPU detection via torch (most reliable on Kaggle) ────────────────────────
try:
    import torch
    USE_GPU = torch.cuda.is_available()
    if USE_GPU:
        print(f'GPU : {torch.cuda.get_device_name(0)}')
        print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
except ImportError:
    USE_GPU = False

print(f'USE_GPU  = {USE_GPU}')
print(f'LightGBM {lgb.__version__} | XGBoost {xgb.__version__}')

# ── Global constants ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
N_FOLDS      = 5
np.random.seed(RANDOM_STATE)

TARGET = 'Irrigation_Need'

CAT_COLS = [
    'Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
    'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region'
]
NUM_COLS = [
    'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
    'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
    'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm'
]

print(f'Numerical features  : {len(NUM_COLS)}')
print(f'Categorical features: {len(CAT_COLS)}')

## Step 1 — Load Data

Auto-detects Kaggle competition path vs local `datasets/` folder.

In [ ]:
# Kaggle mounts competition data at one of these paths
_candidates = [
    '/kaggle/input/playground-series-s6e4',
    '/kaggle/input/competitions/playground-series-s6e4',
]
KAGGLE_INPUT = next((p for p in _candidates if os.path.exists(p)), None)
DATA_DIR     = KAGGLE_INPUT if KAGGLE_INPUT else 'datasets'

print(f'Data directory : {DATA_DIR}')
if os.path.exists('/kaggle/input'):
    print(f'/kaggle/input/ : {os.listdir("/kaggle/input")}')

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
sub   = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print(f'Train : {train.shape}')
print(f'Test  : {test.shape}')
print()
print(train[TARGET].value_counts(normalize=True).round(4))

## Step 2 — Exploratory Data Analysis

- Missing values & dtypes
- Class imbalance visualisation
- Numerical distributions per class
- Pearson correlation heatmap
- Categorical class proportions

In [ ]:
print(f'Missing in train : {train.isnull().sum().sum()}')
print(f'Missing in test  : {test.isnull().sum().sum()}')
print()
print('Dtypes:')
print(train.dtypes)
print()
print('Categorical cardinality:')
for col in CAT_COLS:
    print(f'  {col}: {train[col].nunique()} unique')

In [ ]:
train[NUM_COLS].describe().T\
    .style.background_gradient(cmap='Blues', subset=['mean', 'std'])

In [ ]:
# ~3.3% High -> balanced weighting is critical
palette = {'Low': '#4C9BE8', 'Medium': '#F5A623', 'High': '#E84C4C'}
counts  = train[TARGET].value_counts().reindex(['Low', 'Medium', 'High'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(counts.index, counts.values,
            color=[palette[c] for c in counts.index], edgecolor='white')
axes[0].set_title('Class Counts')
for bar, val in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center')

pcts = counts / counts.sum() * 100
axes[1].pie(pcts.values, labels=pcts.index, autopct='%1.1f%%',
            colors=[palette[c] for c in pcts.index], startangle=140)
axes[1].set_title('Class Share (%)')

plt.suptitle('Target: Irrigation Need', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Overlapping histograms: clear separation = strong predictor
colors_cls = {'Low': '#4C9BE8', 'Medium': '#F5A623', 'High': '#E84C4C'}

fig, axes = plt.subplots(3, 4, figsize=(18, 11))
axes = axes.flatten()

for i, col in enumerate(NUM_COLS):
    for cls in ['Low', 'Medium', 'High']:
        axes[i].hist(train.loc[train[TARGET] == cls, col],
                     bins=40, alpha=0.55, density=True,
                     label=cls, color=colors_cls[cls])
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=7)

for j in range(len(NUM_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
corr = train[NUM_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            annot_kws={'size': 8})
ax.set_title('Pearson Correlation — Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Stacked bars: unequal proportions = predictive signal
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, col in enumerate(CAT_COLS):
    ct = pd.crosstab(train[col], train[TARGET],
                     normalize='index')[['Low', 'Medium', 'High']]
    ct.plot(kind='bar', stacked=True, ax=axes[i],
            color=['#4C9BE8', '#F5A623', '#E84C4C'],
            edgecolor='white', linewidth=0.4)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45, labelsize=7)
    axes[i].legend(fontsize=7)

plt.suptitle('Categorical Feature Class Proportions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 3 — Feature Engineering

Domain-inspired agronomic features:

| Feature | Intuition |
|---------|----------|
| `Water_Stress` | Temp / Moisture — high = crops need water |
| `Evapotranspiration_Proxy` | Sunlight × Temp × (1−Humidity) |
| `Moisture_Deficit` | Distance from field capacity (65%) |
| `Total_Water_Input` | Rainfall + Previous Irrigation |
| `Soil_Quality` | Organic Carbon / EC |

In [ ]:
def engineer_features(df):
    df = df.copy()
    # Water stress & availability
    df['Water_Stress']              = df['Temperature_C'] / (df['Soil_Moisture'] + 1e-3)
    df['Rain_vs_Prev']              = df['Rainfall_mm'] / (df['Previous_Irrigation_mm'] + 1e-3)
    df['Moisture_Deficit']          = 65 - df['Soil_Moisture']
    df['Total_Water_Input']         = df['Rainfall_mm'] + df['Previous_Irrigation_mm']
    df['Prev_Irrig_Moisture_ratio'] = df['Previous_Irrigation_mm'] / (df['Soil_Moisture'] + 1e-3)
    # Atmospheric demand
    df['Temp_Humidity']            = df['Temperature_C'] * (1 - df['Humidity'] / 100)
    df['Evapotranspiration_Proxy'] = df['Sunlight_Hours'] * df['Temp_Humidity']
    df['Wind_Evap']                = df['Wind_Speed_kmh'] * df['Temperature_C'] / (df['Humidity'] + 1)
    df['Rain_per_hour']            = df['Rainfall_mm'] / (df['Sunlight_Hours'] + 1)
    # Soil quality
    df['EC_pH_interaction'] = df['Electrical_Conductivity'] * df['Soil_pH']
    df['Carbon_Moisture']   = df['Organic_Carbon'] * df['Soil_Moisture']
    df['Soil_Quality']      = df['Organic_Carbon'] / (df['Electrical_Conductivity'] + 1e-3)
    return df

train = engineer_features(train)
test  = engineer_features(test)

NEW_FEATURES = [
    'Water_Stress', 'Rain_vs_Prev', 'Temp_Humidity', 'Evapotranspiration_Proxy',
    'Moisture_Deficit', 'EC_pH_interaction', 'Carbon_Moisture', 'Wind_Evap',
    'Rain_per_hour', 'Prev_Irrig_Moisture_ratio', 'Total_Water_Input', 'Soil_Quality'
]
FEATURE_COLS = NUM_COLS + NEW_FEATURES + CAT_COLS
print(f'Features: {len(FEATURE_COLS)} total  '
      f'({len(NUM_COLS)} raw + {len(NEW_FEATURES)} engineered + {len(CAT_COLS)} cat)')

In [ ]:
# Target: string -> int for sklearn
# Categoricals: integer-encoded for LGBM/XGB; CatBoost uses raw strings via Pool
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
target_inv = {v: k for k, v in target_map.items()}

y      = train[TARGET].map(target_map).values
X      = train[FEATURE_COLS].copy()
X_test = test[FEATURE_COLS].copy()

# Fit on combined train+test to avoid unseen-category errors
for col in CAT_COLS:
    le = LabelEncoder()
    le.fit(pd.concat([X[col], X_test[col]]))
    X[col]      = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])

print(f'X      : {X.shape}')
print(f'y      : {y.shape}')
print(f'Classes: {dict(zip(target_map.keys(), np.bincount(y)))}')

## Step 4 — Cross-Validation Setup

**Stratified 5-Fold** preserves class ratios in each fold.  
OOF probabilities are used leak-free for ensemble weight search.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

oof_lgbm  = np.zeros((len(X), 3))
oof_xgb   = np.zeros((len(X), 3))
oof_cat   = np.zeros((len(X), 3))

test_lgbm = np.zeros((len(X_test), 3))
test_xgb  = np.zeros((len(X_test), 3))
test_cat  = np.zeros((len(X_test), 3))

X_arr      = X.values
X_test_arr = X_test.values

print(f'CV: {N_FOLDS}-fold stratified  |  GPU: {USE_GPU}')

## Step 5 — LightGBM

Histogram GBDT. `class_weight='balanced'` handles imbalance.  
GPU: `device='gpu'` — ~3-5x faster on Kaggle T4.

In [ ]:
lgbm_params = {
    'objective':         'multiclass',
    'num_class':         3,
    'metric':            'multi_logloss',
    'n_estimators':      1200,
    'learning_rate':     0.04,
    'num_leaves':        127,
    'max_depth':         -1,
    'min_child_samples': 50,
    'subsample':         0.8,
    'subsample_freq':    1,
    'colsample_bytree':  0.8,
    'reg_alpha':         0.1,
    'reg_lambda':        1.0,
    'class_weight':      'balanced',
    'device':            'gpu' if USE_GPU else 'cpu',
    'random_state':      RANDOM_STATE,
    'n_jobs':            -1,
    'verbose':           -1,
}

lgbm_scores = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_arr, y)):
    X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
    y_tr, y_va = y[tr_idx],     y[va_idx]

    m = lgb.LGBMClassifier(**lgbm_params)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(100, verbose=False),
                     lgb.log_evaluation(-1)])

    oof_lgbm[va_idx] = m.predict_proba(X_va)
    test_lgbm        += m.predict_proba(X_test_arr) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_lgbm[va_idx].argmax(1))
    lgbm_scores.append(score)
    print(f'  Fold {fold+1}: {score:.5f}  (iter {m.best_iteration_})')

lgbm_cv = balanced_accuracy_score(y, oof_lgbm.argmax(1))
print(f'\nLightGBM OOF: {lgbm_cv:.5f}')

## Step 6 — XGBoost

Alternative GBDT. `compute_sample_weight('balanced')` compensates imbalance.  
GPU: `device='cuda'` (XGBoost >= 2.0) — ~4-6x faster.

In [ ]:
sample_weights = compute_sample_weight('balanced', y)

xgb_params = {
    'objective':        'multi:softprob',
    'num_class':        3,
    'eval_metric':      'mlogloss',
    'n_estimators':     1200,
    'learning_rate':    0.04,
    'max_depth':        7,
    'min_child_weight': 10,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'tree_method':      'hist',
    'device':           'cuda' if USE_GPU else 'cpu',
    'random_state':     RANDOM_STATE,
    'n_jobs':           -1,
    'verbosity':        0,
}

xgb_scores = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_arr, y)):
    X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
    y_tr, y_va = y[tr_idx],     y[va_idx]
    sw_tr      = sample_weights[tr_idx]

    m = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=100)
    m.fit(X_tr, y_tr, sample_weight=sw_tr,
          eval_set=[(X_va, y_va)], verbose=False)

    oof_xgb[va_idx] = m.predict_proba(X_va)
    test_xgb        += m.predict_proba(X_test_arr) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_xgb[va_idx].argmax(1))
    xgb_scores.append(score)
    print(f'  Fold {fold+1}: {score:.5f}')

xgb_cv = balanced_accuracy_score(y, oof_xgb.argmax(1))
print(f'\nXGBoost OOF: {xgb_cv:.5f}')

## Step 7 — CatBoost

Native categorical support via ordered target statistics — no encoding leakage.  
GPU: `task_type='GPU'` — biggest speedup of all three models (~10x).

In [ ]:
from catboost import Pool as CatPool

# CatBoost receives raw string categoricals — no label encoding needed
X_cat_str      = train[FEATURE_COLS].copy()
X_test_cat_str = test[FEATURE_COLS].copy()

cat_params = {
    'iterations':            1200,
    'learning_rate':         0.04,
    'depth':                 7,
    'l2_leaf_reg':           3.0,
    'bagging_temperature':   0.5,
    'random_strength':       1.0,
    'auto_class_weights':    'Balanced',
    'loss_function':         'MultiClass',
    'eval_metric':           'Accuracy',
    'early_stopping_rounds': 100,
    'random_seed':           RANDOM_STATE,
    'verbose':               0,
    'task_type':             'GPU' if USE_GPU else 'CPU',
}

cat_scores = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cat_str, y)):
    X_tr_df = X_cat_str.iloc[tr_idx]
    X_va_df = X_cat_str.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    train_pool = CatPool(X_tr_df, label=y_tr, cat_features=CAT_COLS)
    val_pool   = CatPool(X_va_df, label=y_va, cat_features=CAT_COLS)
    test_pool  = CatPool(X_test_cat_str,      cat_features=CAT_COLS)

    m = CatBoostClassifier(**cat_params)
    m.fit(train_pool, eval_set=val_pool, use_best_model=True)

    oof_cat[va_idx] = m.predict_proba(val_pool)
    test_cat        += m.predict_proba(test_pool) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_cat[va_idx].argmax(1))
    cat_scores.append(score)
    print(f'  Fold {fold+1}: {score:.5f}')

cat_cv = balanced_accuracy_score(y, oof_cat.argmax(1))
print(f'\nCatBoost OOF: {cat_cv:.5f}')

## Step 8 — Base Model Comparison

In [ ]:
scores_df = pd.DataFrame({
    'Model':      ['LightGBM', 'XGBoost', 'CatBoost'],
    'OOF_BalAcc': [lgbm_cv, xgb_cv, cat_cv],
}).sort_values('OOF_BalAcc', ascending=False)

print(scores_df.to_string(index=False))
print(f'\nBest: {scores_df.iloc[0]["Model"]}  ->  Optuna tuning')

## Step 9 — Optuna Hyperparameter Tuning

TPE sampler, 50 trials. 3-fold inner CV for speed.  
Objective: maximise OOF Balanced Accuracy.

In [ ]:
skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def lgbm_objective(trial):
    params = {
        'objective':         'multiclass',
        'num_class':         3,
        'metric':            'multi_logloss',
        'n_estimators':      trial.suggest_int('n_estimators', 600, 2000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 63, 255),
        'max_depth':         trial.suggest_int('max_depth', 6, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'subsample_freq':    1,
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_split_gain':    trial.suggest_float('min_split_gain', 0.0, 0.5),
        'class_weight':      'balanced',
        'device':            'gpu' if USE_GPU else 'cpu',
        'random_state':      RANDOM_STATE,
        'n_jobs':            -1,
        'verbose':           -1,
    }
    oof = np.zeros((len(X_arr), 3))
    for tr_idx, va_idx in skf3.split(X_arr, y):
        m = lgb.LGBMClassifier(**params)
        m.fit(X_arr[tr_idx], y[tr_idx],
              eval_set=[(X_arr[va_idx], y[va_idx])],
              callbacks=[lgb.early_stopping(80, verbose=False),
                         lgb.log_evaluation(-1)])
        oof[va_idx] = m.predict_proba(X_arr[va_idx])
    return balanced_accuracy_score(y, oof.argmax(1))

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study.optimize(lgbm_objective, n_trials=50, show_progress_bar=True)

print(f'\nBest score : {study.best_value:.5f}')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

## Step 10 — Retrain Tuned LightGBM (5-Fold)

In [ ]:
best_params = study.best_params.copy()
best_params.update({
    'objective': 'multiclass', 'num_class': 3, 'metric': 'multi_logloss',
    'class_weight': 'balanced', 'subsample_freq': 1,
    'device': 'gpu' if USE_GPU else 'cpu',
    'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbose': -1,
})

oof_tuned  = np.zeros((len(X_arr), 3))
test_tuned = np.zeros((len(X_test_arr), 3))
tuned_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_arr, y)):
    X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
    y_tr, y_va = y[tr_idx],     y[va_idx]

    m = lgb.LGBMClassifier(**best_params)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(100, verbose=False),
                     lgb.log_evaluation(-1)])

    oof_tuned[va_idx] = m.predict_proba(X_va)
    test_tuned        += m.predict_proba(X_test_arr) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_tuned[va_idx].argmax(1))
    tuned_scores.append(score)
    print(f'  Fold {fold+1}: {score:.5f}  (iter {m.best_iteration_})')

tuned_cv = balanced_accuracy_score(y, oof_tuned.argmax(1))
print(f'\nTuned LightGBM OOF: {tuned_cv:.5f}')

## Step 11 — Ensemble Weight Search

Grid search over weight simplex (step 0.1) for all 4 OOF outputs.  
Maximises OOF Balanced Accuracy — no leakage.

In [ ]:
from itertools import product as iproduct

best_ens_score = 0.0
best_weights   = (0.4, 0.2, 0.2, 0.2)

steps = np.arange(0, 1.01, 0.1)
for w1, w2, w3 in iproduct(steps, steps, steps):
    w4 = round(1.0 - w1 - w2 - w3, 5)
    if w4 < 0:
        continue
    ens = w1*oof_lgbm + w2*oof_xgb + w3*oof_cat + w4*oof_tuned
    s   = balanced_accuracy_score(y, ens.argmax(1))
    if s > best_ens_score:
        best_ens_score = s
        best_weights   = (w1, w2, w3, w4)

w1, w2, w3, w4 = best_weights
print(f'Best weights: LGBM={w1:.1f} XGB={w2:.1f} CAT={w3:.1f} LGBM_tuned={w4:.1f}')
print(f'Ensemble OOF: {best_ens_score:.5f}')

test_ensemble = w1*test_lgbm + w2*test_xgb + w3*test_cat + w4*test_tuned

## Step 12 — Final Score Summary

In [ ]:
final_df = pd.DataFrame({
    'Model': ['LightGBM', 'XGBoost', 'CatBoost', 'LightGBM (tuned)', 'Ensemble'],
    'OOF_Balanced_Acc': [lgbm_cv, xgb_cv, cat_cv, tuned_cv, best_ens_score],
}).sort_values('OOF_Balanced_Acc', ascending=False).reset_index(drop=True)

print('=== Final CV Scores ===')
print(final_df.to_string(index=False))

## Step 13 — Create Submission

In [ ]:
if best_ens_score >= tuned_cv:
    final_proba = test_ensemble
    chosen      = 'Ensemble'
else:
    final_proba = test_tuned
    chosen      = 'Tuned LightGBM'

print(f'Using: {chosen}')

submission = pd.DataFrame({
    'id':              test['id'].values,
    'Irrigation_Need': [target_inv[i] for i in final_proba.argmax(1)]
})

OUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
submission.to_csv(f'{OUT_DIR}/submission.csv', index=False)
print(f'Saved: {OUT_DIR}/submission.csv  ({len(submission)} rows)')
print(submission['Irrigation_Need'].value_counts())
submission.head()

In [ ]:
assert list(submission.columns) == ['id', 'Irrigation_Need']
assert len(submission) == len(sub), f'{len(submission)} != {len(sub)}'
assert set(submission['Irrigation_Need'].unique()) <= {'Low','Medium','High'}
print('All checks passed!')